In [2]:
!pip install torch

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 4.8 MB/s eta 0:00:26
    --------------------------------------- 2.6/124.1 MB 8.2 MB/s eta 0:00:15
   - -------------------------------------- 5.5/124.1 MB 10.7 MB/s eta 0:00:12
   -- ------------------------------------- 8.4/124.1 MB 11.5 MB/s eta 0:00:11
   --- ------------------------------------ 10.7/124.1 MB 11.8 MB/s eta 0:00:10
   ---- ----------------------------------- 13.6/124.1 MB 12.6 MB/s eta 0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn

In [ ]:
class BaselineLSTM(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.channels = channels

        self.pooling = nn.AvgPool1d(kernel_size=5, stride=5)

        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=self.channels, out_channels=32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(in_channels=64, out_channels=64, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(input_size=64, hidden_size=64, num_layers=1, batch_first=True, bidirectional=False)

        self.lin_head = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = self.pooling(x)
        x = self.cnn(x)
        x = x.transpose(1, 2)
        _, (h,) = self.lstm(x)
        x = h[-1]
        pred = self.lin_head(x)
        return pred.squeeze(-1)